FOR ARYA & LI, PLEASE ONLY RUN COLUMN 120 (AUDIO SIMULATION), DON'T FORGET TO CHANGE THE FILE DIRECTORY IF NECESSARY

In [ ]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

In [ ]:
import os
import pandas as pd

# 1. 基础路径保持不变
path_base = r"<LOCAL_PROJECT_ROOT>"

# --- 关键修正：这里加了一个嵌套的文件夹名，匹配你的真实路径 ---
path_data = os.path.join(path_base, "Respiratory_Sound_Database", "Respiratory_Sound_Database", "audio_and_txt_files")
path_diagnosis = os.path.join(path_base, "csv_data", "data.csv") 

# 2. 读取数据，保持 header=0 解决之前的文字报错
diagnosis_df = pd.read_csv(
    path_diagnosis,
    names=['pid', 'disease'],
    header=0 
)

# 3. 处理 'sc' 导致的转换报错，保持健壮性
diagnosis_df['pid'] = pd.to_numeric(diagnosis_df['pid'], errors='coerce') 
diagnosis_df = diagnosis_df.dropna(subset=['pid']) 
diagnosis_df['pid'] = diagnosis_df['pid'].astype('int32')

# 4. 获取标注文件
files = [f.split('.')[0] for f in os.listdir(path_data) if f.endswith('.txt')]

def getFilenameInfo(file):
    return file.split('_')

files_data = []

# 5. 循环读取，使用 os.path.join 确保路径中间有斜杠
for file in files:
    annot_df = pd.read_csv(
        os.path.join(path_data, file + '.txt'), 
        sep='\t',
        names=['start', 'end', 'crackles', 'wheezes']
    )

    name_data = getFilenameInfo(file)
    annot_df['pid'] = int(name_data[0])
    annot_df['mode'] = name_data[-2]
    annot_df['filename'] = file

    files_data.append(annot_df)

# 6. 合并数据
files_df = pd.concat(files_data, ignore_index=True)
files_df['pid'] = files_df['pid'].astype('int32')

final_df = pd.merge(
    files_df,
    diagnosis_df,
    on='pid',
    how='left'
)

# 7. 打印结果
print(final_df.head())
print("\nColumns:\n", final_df.columns)
print("\nMissing disease labels:", final_df['disease'].isna().sum())

In [ ]:
# Making the merging data (patient data + diagnosis) directory and check if there's available merging data already
isCsvDataExist = os.path.exists('csv_data')
if(not isCsvDataExist):
    os.makedirs('csv_data')
    files_df.to_csv('csv_data/data.csv',index=False)
isCsvDataExist

In [ ]:
# Getting the raw audio from the data
def getPureSample(raw_data,start,end,sr):
    max_ind = len(raw_data) 
    start_ind = min(int(start * sr), max_ind)
    end_ind = min(int(end * sr), max_ind)
    return raw_data[start_ind: end_ind]

In [ ]:
# Making processed audio files in the directory and check for availability
isProcessedAudioFilesExist = os.path.exists('processed_audio_files')
isProcessedAudioFilesExistFlag = False

if(not isProcessedAudioFilesExist):
    os.makedirs('processed_audio_files')
    isProcessedAudioFilesExistFlag = True
isProcessedAudioFilesExist

In [ ]:
# Checking one data in the dataframe
import librosa as lb
for index,row in final_df.iterrows():
    print("Index:",index)
    print("DataRow:\n",row)
    audio_file_loc=path_data + row['filename'] + '.wav'
    audioArr,sampleRate=lb.load(audio_file_loc)
    print("sr:",sampleRate)
    break

In [ ]:
# Changing the audio files into 6 seconds to avoid imbalanced data
# ONLY RUN THIS PART IF THE PREVIOUS 'processed_audio_data' is FALSE
# If 'processed_audio_data' is TRUE, no need to run this part

import soundfile as sf
import librosa as lb

final_df['start'] = final_df['start'].astype(float)
final_df['end'] = final_df['end'].astype(float)

if(isProcessedAudioFilesExistFlag):
    i, cnt = 0, 0

    for index, row in final_df.iterrows():
        duration = 6
        start = row['start']
        end = row['end']
        filename = row['filename']
        
        if end - start > duration:
            end = start + duration
        
        audio_file_loc = path_data + filename + '.wav'

        if index > 0 and final_df.iloc[index - 1]['filename'] == filename:
            i += 1
        else:
            i = 0

        out_filename = f"{filename}_{i}.wav"
        save_path = 'processed_audio_files/' + out_filename
        cnt += 1

        print(f"Processing file: {audio_file_loc}")
        print(f"Saving to: {save_path}")
        
        audioArr, sampleRate = lb.load(audio_file_loc)
        pureSample = getPureSample(audioArr, start, end, sampleRate)
        
        reqLen = 6 * sampleRate
        padded_data = lb.util.pad_center(data=pureSample, size=reqLen)
        
        sf.write(file=save_path, data=padded_data, samplerate=sampleRate)
    
    print('Total Files Processed:', cnt)


In [ ]:
# Checking the dataframe

def extractId(filename):
    return int(filename.split('_')[0])

path = 'processed_audio_files/'
files = os.listdir(path)
files_df = pd.DataFrame(columns=['pid', 'filename'])

for i, f in enumerate(files):
    files_df.loc[i, 'pid'] = extractId(f)
    files_df.loc[i, 'filename'] = f

files_df.head()


In [ ]:
# Eliminate duplicate dataframe

files_df = files_df[
    ~files_df['filename'].str.contains('_noise|_pitch_|_speed_')
]

In [ ]:
# Ensuring the same datatype
files_df.pid = files_df.pid.astype('int64')

# Attach disease labels
files_df = pd.merge(
    files_df,
    final_df[['pid', 'disease']].drop_duplicates(),
    on='pid',
    how='left'
)


In [ ]:
# def extractId(filename):
#     return filename.split('_')[0]

# path='processed_audio_files/'
# files = os.listdir(path)
# files_df = pd.DataFrame(columns=['pid', 'filename'])

# for i, f in enumerate(files):
#     files_df.loc[i, 'pid'] = extractId(f)
#     files_df.loc[i, 'filename'] = f

# files_df.head()

In [ ]:
# Defining augmentation data function to avoid imbalanced data

# Adding noise
def aug_noise(audio, noise_level = 0.005):
    noise = np.random.normal(0, noise_level, len(audio))
    return audio + noise

# Pitch audio
def aug_pitch(audio, sample_rate, n_steps):
    return lb.effects.pitch_shift(audio, sr=sample_rate, n_steps=n_steps)

# Speed audio
def aug_speed(audio, speed_factor):
    return lb.effects.time_stretch(audio, rate=speed_factor)

# Fixing the sample's duration into 6 seconds
def adjust_audio_duration(audio, sr, max_duration=6.0):
    max_length = int(sr * max_duration)  # Maximum duration of the sample
    if len(audio) < max_length:
        # Padding if sample < 6 seconds
        return lb.util.fix_length(data=audio, size=max_length)
    else:
        # Cutting if sample > 6 seconds
        return audio[:max_length]

In [ ]:
# Classes that will be augmented
augment_class = ['Healthy', 'COPD', 'URTI', 'Bronchiectasis', 'Bronchiolitis', 'Pneumonia']
filtered_data = files_df[files_df['disease'].isin(augment_class)]

print(filtered_data)

In [ ]:
for index, row in filtered_data.iterrows():
    filename, ext = os.path.splitext(row['filename'])
    if ext != '.wav':  # Pastikan hanya ada satu ekstensi .wav
        filename += '.wav'
    # filename = row['filename']
    print(filename)

In [ ]:
import soundfile as sf
augmented_files = []
max_duration = 6.0
for index, row in filtered_data.iterrows():
    filename = row['filename']
    # filename, ext = os.path.splitext(row['filename'])
    # if ext != '.wav':  # Pastikan hanya ada satu ekstensi .wav
    #     filename += '.wav'

    file_path = os.path.join(path, filename)

    # Debugging file path
    print(f"Trying to load file: {file_path}")
    
    # See if file path exist
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}, skipping...")
        continue

    audio, sr = lb.load(file_path)

    base_filename, _ = os.path.splitext(filename)

    # Adding noise
    audio_with_noise = aug_noise(audio)
    noise_file_path = f"processed_audio_files/{base_filename}_noise.wav"
    sf.write(noise_file_path, audio_with_noise, sr)
    augmented_files.append({'filename': f"{base_filename}_noise.wav", 'disease': row['disease']})

    # Tuning pitch up
    audio_with_pitch_up = aug_pitch(audio, sr, n_steps=2)
    pitch_up_file_path = f"processed_audio_files/{base_filename}_pitch_up.wav"
    sf.write(pitch_up_file_path, audio_with_pitch_up, sr)
    augmented_files.append({'filename': f"{base_filename}_pitch_up.wav", 'disease': row['disease']})

    # Tuning pitch down
    audio_with_pitch_down = aug_pitch(audio, sr, n_steps=-2)
    pitch_down_file_path = f"processed_audio_files/{base_filename}_pitch_down.wav"
    sf.write(pitch_down_file_path, audio_with_pitch_down, sr)
    augmented_files.append({'filename': f"{base_filename}_pitch_down.wav", 'disease': row['disease']})

    # Tuning audio speed (faster)
    audio_with_speed_up = aug_speed(audio, speed_factor=1.2)
    speed_up_file_path = f"processed_audio_files/{base_filename}_speed_up.wav"
    sf.write(speed_up_file_path, audio_with_speed_up, sr)
    augmented_files.append({'filename': f"{base_filename}_speed_up.wav", 'disease': row['disease']})

   # Tuning audio speed (slower)
    audio_with_speed_down = aug_speed(audio, speed_factor=0.8)
    speed_down_file_path = f"processed_audio_files/{base_filename}_speed_down.wav"
    sf.write(speed_down_file_path, audio_with_speed_down, sr)
    augmented_files.append({'filename': f"{base_filename}_speed_down.wav", 'disease': row['disease']})


# Adding augmented data to the dataset
augmented_df = pd.DataFrame(augmented_files)
augmented_df['pid'] = augmented_df['filename'].apply(lambda x: int(os.path.basename(x).split('_')[0]))
data = pd.concat([files_df, augmented_df], ignore_index=True)

In [ ]:
# See how much data each class has (has to be the same amount)
print(data['disease'].value_counts())


In [ ]:
# To ensure the data sample's duration is 6 seconds after augmentation process

processed_audio_dir = "processed_audio_files"

if not os.path.exists(processed_audio_dir):
    print(f"Directory '{processed_audio_dir}' does not exist.")
else:
    cnt = 0
    for filename in os.listdir(processed_audio_dir):
        if filename.endswith('.wav'):
            file_path = os.path.join(processed_audio_dir, filename)
            print(f"Processing file: {file_path}")

            try:
                audio, sr = lb.load(file_path, sr=None)
                adjusted_audio = adjust_audio_duration(audio, sr, max_duration=6.0)

                # Menyimpan audio yang sudah disesuaikan
                save_path = file_path
                sf.write(save_path, adjusted_audio, samplerate=sr)
                cnt += 1
                print(f"Saved adjusted file: {save_path}")
                
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")

    print(f"Total Files Processed: {cnt}")

In [ ]:
# Dividing data into 80% train and 20% temp

from sklearn.model_selection import train_test_split
Xtrain,Xval,ytrain,yval=train_test_split(data,data.disease,stratify=data.disease,random_state=42,test_size=0.20)

In [ ]:
Xtrain.disease.value_counts()/Xtrain.shape[0]
Xval.disease.value_counts()/Xval.shape[0]

Xtrain.head()
ytrain.head()
Xval.head()
yval.head()

In [ ]:
# Creating the training and validation directory

TrainingFileExist = os.path.exists('training')
ValidationFileExist = os.path.exists('validation')
if(not TrainingFileExist):
    os.makedirs('training')
    Xtrain.to_csv('training/train.csv')
if(not ValidationFileExist):
    os.makedirs('validation')
    Xval.to_csv('validation/val.csv')

In [ ]:
# Reading the training and validation .csv files
train=pd.read_csv('training/train.csv')
val=pd.read_csv('validation/val.csv')
train.head()

In [ ]:
ytrain=train.disease
yval=val.disease
print(yval)

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Encode the data
le=LabelEncoder()
ytrain=le.fit_transform(ytrain)
yval=le.transform(yval)
print(yval)

In [ ]:
import librosa

# Feature Extraction
def featureExtraction(path):
    soundArr,sample_rate=lb.load(path)
    mel_feat = librosa.feature.melspectrogram(y=soundArr, sr=sample_rate)
    mSpec = librosa.power_to_db(mel_feat,ref=np.max)

    return mSpec

In [ ]:
# Training
root='processed_audio_files/'
mSpec = []

for idx,row in train.iterrows():
    path=root + row['filename']
    a=featureExtraction(path)
    mSpec.append(a)
    
mSpec_train=np.array(mSpec)

In [ ]:
# Save the variables into .txt so we dont have to extract again
# Convert 3d array to 2d to save it
# While loading need to reshape back to 3d

mSpec_reshaped = mSpec_train.reshape(mSpec_train.shape[0], -1)

print(mSpec_reshaped)

In [ ]:
# Save into .txt file
np.savetxt('mSpec_variable_train.txt', mSpec_reshaped)

In [ ]:
# Import variable from .txt file
mSpec_train = np.loadtxt('mSpec_variable_train.txt')

In [ ]:
mSpec_train = mSpec_train.reshape(
    mSpec_train.shape[0], mSpec_train.shape[1] // 259 , 259)

In [ ]:
print(mSpec_train.shape)

In [ ]:
# Validation
mSpec= []

for idx,row in val.iterrows():
    path=root + row['filename']
    a=featureExtraction(path)
    mSpec.append(a)
  
mSpec_val=np.array(mSpec)

In [ ]:
mSpec_reshaped = mSpec_val.reshape(mSpec_val.shape[0], -1)

print(mSpec_reshaped)

In [ ]:
np.savetxt('mSpec_variable_val.txt', mSpec_reshaped)

In [ ]:
mSpec_val = np.loadtxt('mSpec_variable_val.txt')

In [ ]:
mSpec_val = mSpec_val.reshape(
    mSpec_val.shape[0], mSpec_val.shape[1] // 259 , 259)

In [ ]:
print(mSpec_val.shape)

In [ ]:
callback = [
    # tf.keras.callbacks.EarlyStopping(patience=5),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1,
                              patience=3, min_lr=0.00001,mode='min')
]

MODEL MEL SPECTROGRAM

In [ ]:
mSpec_input=keras.layers.Input(shape=(128,259,1),name="mSpecInput")
x=keras.layers.Conv2D(32,5,strides=(2,3),padding='same')(mSpec_input)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(keras.activations.relu)(x)
x=keras.layers.MaxPooling2D(pool_size=2,padding='valid')(x)

x=keras.layers.Conv2D(64,3,strides=(2,2),padding='same')(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(keras.activations.relu)(x)
x=keras.layers.MaxPooling2D(pool_size=2,padding='valid')(x)

x=keras.layers.Conv2D(96,2,padding='same')(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(keras.activations.relu)(x)
x=keras.layers.MaxPooling2D(pool_size=2,padding='valid')(x)

x=keras.layers.Conv2D(128,2,padding='same')(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(keras.activations.relu)(x)
mSpec_output=keras.layers.GlobalMaxPooling2D()(x)

mSpec_model=keras.Model(mSpec_input, mSpec_output, name="mSpecModel")

input_mSpec=keras.layers.Input(shape=(128,259,1),name="mspec")
mSpec=mSpec_model(input_mSpec)

concat=keras.layers.concatenate([mSpec])
hidden=keras.layers.Dropout(0.2)(concat)
hidden=keras.layers.Dense(50,activation='relu')(concat)
hidden=keras.layers.Dropout(0.3)(hidden)
hidden=keras.layers.Dense(25,activation='relu')(hidden)
hidden=keras.layers.Dropout(0.3)(hidden)
output=keras.layers.Dense(6,activation='softmax')(hidden)

mSpec_model = keras.Model([input_mSpec], output, name="mSpec_model")

mSpec_model.summary()

In [ ]:
from tensorflow.keras.optimizers import Nadam
import datetime

# Training model Mel Spectrogram
mSpec_model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer = Nadam(0.001),
    metrics = ['accuracy']

)

start = datetime.datetime.now()
history_mspec = mSpec_model.fit(
        {"mspec":mSpec_train},
        ytrain,
        validation_data=({"mspec":mSpec_val},yval),
        epochs=200,verbose=1,
        callbacks=callback
)
end = datetime.datetime.now()

print(f'Total Training Time: {end-start}')

In [ ]:
from tensorflow.keras.models import save_model
# Saving the model's weight
save_model(mSpec_model, "melspectogramTrained.h5")

In [ ]:
# Plotting the loss and accuracy of the model
plt.plot(history_mspec.history['loss'], label = 'train_loss')
plt.plot(history_mspec.history['val_loss'], label ='val_loss')
plt.title("Mel Spectrogram Loss Over Epoch")
plt.legend()
plt.show()

plt.plot(history_mspec.history['accuracy'], label = 'train accuracy')
plt.plot(history_mspec.history['val_accuracy'], label ='val accuracy')
plt.title("Mel Spectrogram Accuracy Over Epoch")
plt.legend()
plt.show()

In [ ]:
mSpec_model.evaluate({"mspec":mSpec_val},yval)

In [ ]:
# Classification report
ypred = mSpec_model.predict({"mspec":mSpec_val})

yPred = []
for i in ypred:
    yPred.append(np.argmax(i))
yPred

print('\nClassification Report\n')
from sklearn.metrics import classification_report
print(classification_report(yval, yPred, target_names=['Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'Pneumonia', 'URTI']))

In [ ]:
import os
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import librosa as lb
import numpy as np
import pandas as pd

# Plot confusion matrix model Mel Spectrogram
df = pd.read_csv("<LOCAL_PROJECT_ROOT>\csv_data\label_mapping_new.csv")
label_mapping = dict(zip(df['filename'], df['label']))

directory = os.path.join(os.getcwd(), "processed_audio_files")
all_files = os.listdir(directory)


test_files = [(f, label_mapping[f]) for f in all_files if f.endswith('.wav') and f in label_mapping]

y_true = []
y_pred = []

def featureExtractionNN(path):
    soundArr, sample_rate = lb.load(path)
    mel_feat = librosa.feature.melspectrogram(y=soundArr, sr=sample_rate)
    mSpec = librosa.power_to_db(mel_feat,ref=np.max)

    return mSpec

diseaseArray = ['Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'Pneumonia', 'URTI']

for file_name, true_label in test_files:
    soundFilePath = os.path.join(directory, file_name)
    mSpec_test = featureExtractionNN(soundFilePath)

    mSpec = np.array([mSpec_test])


    result = mSpec_model.predict({"mspec": mSpec}).flatten()
    predicted_label = diseaseArray[np.argmax(result)]

    y_true.append(true_label)
    y_pred.append(predicted_label)

cm_mspec = confusion_matrix(y_true, y_pred, labels=diseaseArray)
cmd = ConfusionMatrixDisplay(cm_mspec, display_labels=diseaseArray)
cmd.plot(xticks_rotation='vertical')
plt.show()


In [ ]:
# ICBHI Report
sensitivity_list = []
specificity_list = []
harmonic_score_list = []

for i in range(len(cm_mspec)):
    TP = cm_mspec[i, i] 
    FN = np.sum(cm_mspec[i, :]) - TP  
    FP = np.sum(cm_mspec[:, i]) - TP  
    TN = np.sum(cm_mspec) - (TP + FN + FP)  

    sensitivity = TP / (TP + FN) if (TP + FN) != 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) != 0 else 0
    harmonic_score = TP / (TP + FN) if (TP + FN) != 0 else 0 
    avg_score = (sensitivity + specificity) / 2 

    sensitivity_list.append(sensitivity)
    specificity_list.append(specificity)
    harmonic_score_list.append(harmonic_score)

avg_sensitivity = np.mean(sensitivity_list)
avg_specificity = np.mean(specificity_list)
avg_harmonic_score = np.mean(harmonic_score_list)
avg_avg_score = np.mean([(sens + spec) / 2 for sens, spec in zip(sensitivity_list, specificity_list)])

print(f"Average Sensitivity: {avg_sensitivity:.4f}")
print(f"Average Specificity: {avg_specificity:.4f}")
print(f"Average Harmonic Score: {avg_harmonic_score:.4f}")
print(f"Average Score: {avg_avg_score:.4f}")

In [ ]:
import os
import openai
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DEEPBREATH_LLM_API_KEY")
base_url = os.getenv("DEEPBREATH_LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/")

if not api_key:
    raise ValueError("请设置 DEEPBREATH_LLM_API_KEY 环境变量")
# 初始化客户端
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

In [ ]:
import os
import pandas as pd
import openai
from openai import OpenAI
from dotenv import load_dotenv
# from google import genai

load_dotenv()


path_base = r"<LOCAL_PROJECT_ROOT>"
path_data = os.path.join(path_base, "Respiratory_Sound_Database", "Respiratory_Sound_Database", "audio_and_txt_files")
path_diagnosis = os.path.join(path_base, "Respiratory_Sound_Database", "Respiratory_Sound_Database", "patient_diagnosis.csv")


diag_df = pd.read_csv(path_diagnosis, names=['pid', 'disease'], header=None)
diag_df['pid'] = pd.to_numeric(diag_df['pid'], errors='coerce')
diag_df = diag_df.dropna(subset=['pid']).astype({'pid': 'int32'})

files = [f.split('.')[0] for f in os.listdir(path_data) if f.endswith('.txt')]
files_data = []
for f in files:
    temp_df = pd.read_csv(os.path.join(path_data, f + '.txt'), sep='\t', names=['start', 'end', 'crackles', 'wheezes'])
    temp_df['pid'] = int(f.split('_')[0])
    files_data.append(temp_df)

final_df = pd.merge(pd.concat(files_data, ignore_index=True), diag_df, on='pid', how='left')
final_df['pid'] = final_df['pid'].astype('int32')




client = OpenAI(
    api_key=os.getenv("DEEPBREATH_LLM_API_KEY"),
    base_url=os.getenv("DEEPBREATH_LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/")
)
target_model = "glm-4-flash"
print(f"--- Using Model: {target_model} ---")

def generate_patient_report(row):
    pid = row['pid']
    disease = row['disease']
    # 汇总该病人的所有呼吸周期特征
    c_total = final_df[final_df['pid'] == pid]['crackles'].sum()
    w_total = final_df[final_df['pid'] == pid]['wheezes'].sum()


    prompt = f"""
    Context: UCL Respiratory Research Project.
    Patient ID: {pid} | Clinical Diagnosis: {disease}
    Observations: {c_total} crackles and {w_total} wheezes recorded across all cycles.
    
    Task: Provide a professional clinical summary (~100 words) and 2 specific suggestions for this condition.
    """
    
    try:

        response = client.chat.completions.create(
            model=target_model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"AI Error: {str(e)}"



In [ ]:
import os
import threading
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import ttk, scrolledtext, filedialog, messagebox
from dotenv import load_dotenv
from openai import OpenAI
import librosa
import librosa.display
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import pygame
from tensorflow.keras.models import load_model


load_dotenv()
pygame.mixer.init()

# ============================================================
# 1. 配色
# ============================================================
COLOR_UCL_BLUE = "#003B71"
COLOR_ACCENT   = "#00A8FF"
COLOR_BG       = "#F5F6FA"
COLOR_SIDEBAR  = "#2F3640"
COLOR_CARD     = "#FFFFFF"
COLOR_SUCCESS  = "#44BD32"
COLOR_WARN     = "#E1B12C"
COLOR_DANGER   = "#E74C3C"

# ============================================================
# 2. 加载 CNN 模型 & 疾病标签
# ============================================================
DISEASE_CLASSES = ['Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'Pneumonia', 'URTI']
mSpec_model     = load_model("melspectogramTrained.h5")
print(f"[CNN] Model loaded. Classes: {DISEASE_CLASSES}")

# ============================================================
# 3. 加载呼吸数据库（6898 条记录）
# ============================================================
PATH_BASE      = r"<LOCAL_PROJECT_ROOT>"
PATH_DATA      = os.path.join(PATH_BASE, "Respiratory_Sound_Database",
                               "Respiratory_Sound_Database", "audio_and_txt_files")
PATH_DIAGNOSIS = os.path.join(PATH_BASE, "Respiratory_Sound_Database",
                               "Respiratory_Sound_Database", "patient_diagnosis.csv")

# 默认浏览目录：用户在 STEP 1 点击 Browse 时直接打开此目录
PATH_AUDIO_DIR =  os.path.join(PATH_BASE, "processed_audio_files")

try:
    diag_df = pd.read_csv(PATH_DIAGNOSIS, names=['pid', 'disease'], header=None)
    diag_df['pid'] = pd.to_numeric(diag_df['pid'], errors='coerce')
    diag_df = diag_df.dropna(subset=['pid']).astype({'pid': 'int32'})

    files_data = []
    for fname in [x for x in os.listdir(PATH_DATA) if x.endswith('.txt')]:
        temp = pd.read_csv(os.path.join(PATH_DATA, fname), sep='\t',
                           names=['start', 'end', 'crackles', 'wheezes'])
        temp['pid'] = int(fname.split('_')[0])
        files_data.append(temp)

    final_df = pd.merge(pd.concat(files_data, ignore_index=True),
                        diag_df, on='pid', how='left')
    final_df['pid'] = final_df['pid'].astype('int32')
    print(f"[DB] Loaded {len(final_df)} respiratory cycles.")
except Exception as e:
    final_df = pd.DataFrame()
    print(f"[DB] Warning - database not loaded: {e}")


def get_clinical_benchmarks():
    """
    计算全库各病种平均 crackles/wheezes（每个呼吸周期）。
    返回 (prevalence_series, benchmarks_df, total_patients)。
    在 LLM prompt 中提供循证对比基准。
    """
    if final_df.empty:
        return {}, pd.DataFrame(), 0
    total_p    = final_df['pid'].nunique()
    prevalence = (final_df.groupby('pid')['disease'].first()
                  .value_counts() / total_p * 100).round(2)
    benchmarks = final_df.groupby('disease').agg(
        {'crackles': 'mean', 'wheezes': 'mean'}).round(2)
    return prevalence, benchmarks, total_p


def get_patient_sounds(pid):
    """
    根据 pid 查询该病人所有呼吸周期的 crackles/wheezes 总计。
    数据库不可用或 pid 不存在时返回 (0, 0)。
    """
    if final_df.empty or pid is None:
        return 0, 0
    rows = final_df[final_df['pid'] == pid]
    return int(rows['crackles'].sum()), int(rows['wheezes'].sum())

# ============================================================
# 4. 初始化智谱 AI 
# ============================================================
client    = OpenAI(
    api_key  = os.getenv("DEEPBREATH_LLM_API_KEY"),
    base_url = os.getenv("DEEPBREATH_LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/")
)
LLM_MODEL = "glm-4-flash"
print(f"[LLM] Client ready. Model: {LLM_MODEL}")

# ============================================================
# 5. 音频目录常量
# ============================================================

# ============================================================
# 6. 工具函数
# ============================================================
def safe_text(obj):
    """安全转 UTF-8 字符串，防止 Windows 编码崩溃"""
    try:
        return str(obj).encode('utf-8', 'ignore').decode('utf-8') if obj else ""
    except:
        return ""


def extract_pid_from_filename(filename):
    """
    从文件名解析 pid。
    格式：{pid}_{...}.wav，例如 '104_1b1_Al_sc_Litt3200_0.wav' -> 104
    解析失败返回 None。
    """
    try:
        return int(os.path.basename(filename).split('_')[0])
    except:
        return None


def ground_truth_from_db(pid):
    """
    根据 pid 从数据库中查询该病人的真实疾病标签。
    用于在预测结果中显示正误对比（替代原来从下拉框 key 读取的方式）。
    找不到则返回 None（不显示对比，不影响预测流程）。
    """
    if final_df.empty or pid is None:
        return None
    rows = final_df[final_df['pid'] == pid]['disease'].dropna()
    return rows.iloc[0] if not rows.empty else None


def show_plot_in_frame(fig, frame):
    """将 matplotlib Figure 嵌入 tkinter Frame（自动清空旧内容，释放内存）"""
    for w in frame.winfo_children():
        w.destroy()
    canvas = FigureCanvasTkAgg(fig, master=frame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
    plt.close(fig)


def extract_mel_spectrogram(path):
    """
    提取 Mel 频谱图（与 CNN 训练特征提取完全一致）。
    返回 (mSpec_dB, audio_array, sample_rate)。
    """
    audio, sr = librosa.load(path)
    mel_feat  = librosa.feature.melspectrogram(y=audio, sr=sr)
    mSpec_dB  = librosa.power_to_db(mel_feat, ref=np.max)
    return mSpec_dB, audio, sr

# ============================================================
# 7. 全局状态
# ============================================================
cnn_result = {
    "predicted_disease": None,   # Top-1 疾病名
    "confidence":        None,   # Top-1 置信度 (0.0~1.0)
    "runner_up_disease": None,   # Top-2 疾病名（鉴别诊断）
    "runner_up_conf":    None,   # Top-2 置信度
    "all_probs":         None,   # 完整 6 类概率向量
    "pid":               None,   # 从文件名解析的 pid（可能为 None）
    "ground_truth":      None,   # 从数据库按 pid 反查的真实标签（可能为 None）
    "audio_path":        None,   # 音频完整路径
}

# ============================================================
# 8. CNN 推理核心函数
# ============================================================
def run_cnn_inference(full_path, ground_truth_label, root_instance):
    """
    后台线程中执行 CNN 推理流程：
      1. 提取 Mel 频谱图
      2. 绘制波形图 + 频谱图（嵌入 GUI）
      3. 运行 CNN，获取 6 类概率
      4. 绘制置信度条形图（高亮 Top-1）
      5. 写入 cnn_result 全局字典，解锁 Step 3 按钮
    通过 root.after() 回调主线程更新 UI（tkinter 非线程安全）
    """
    try:
        mSpec, audio, sr = extract_mel_spectrogram(full_path)

        # 1. 波形图
        fig_wave = plt.figure(figsize=(5, 2.2))
        librosa.display.waveshow(audio, sr=sr, color=COLOR_ACCENT)
        plt.title("Waveform", fontsize=9)
        plt.tight_layout()
        root_instance.after(0, lambda f=fig_wave: show_plot_in_frame(f, waveform_frame))

        # 2. Mel 频谱图
        fig_spec = plt.figure(figsize=(5, 2.2))
        librosa.display.specshow(mSpec, sr=sr, x_axis='time', y_axis='mel', cmap='magma')
        plt.title("Mel Spectrogram", fontsize=9)
        plt.colorbar(format='%+2.0f dB')
        plt.tight_layout()
        root_instance.after(0, lambda f=fig_spec: show_plot_in_frame(f, spectrogram_frame))

        # 3. CNN 推理
        mSpec_input = np.expand_dims(mSpec, axis=0)   # shape: (1, 128, 259)
        probs       = mSpec_model.predict({"mspec": mSpec_input}).flatten()
        idx1        = np.argmax(probs)
        idx2        = np.argsort(probs)[-2]

        # 4. 写入全局状态（CNN -> LLM 数据桥梁）
        cnn_result.update({
            "predicted_disease": DISEASE_CLASSES[idx1],
            "confidence":        float(probs[idx1]),
            "runner_up_disease": DISEASE_CLASSES[idx2],
            "runner_up_conf":    float(probs[idx2]),
            "all_probs":         probs,
            "pid":               extract_pid_from_filename(full_path),
            "ground_truth":      ground_truth_label,
            "audio_path":        full_path,
        })

        # 5. 置信度条形图（Top-6，UCL蓝高亮 Top-1）
        fig_bar = plt.figure(figsize=(5, 2.5))
        bar_colors = [COLOR_UCL_BLUE if i == idx1 else '#AECCE4' for i in range(6)]
        plt.barh(DISEASE_CLASSES, probs * 100, color=bar_colors)
        plt.xlabel("Confidence (%)", fontsize=8)
        plt.title("CNN Prediction Confidence", fontsize=9)
        plt.xlim(0, 100)
        plt.tight_layout()
        root_instance.after(0, lambda f=fig_bar: show_plot_in_frame(f, confidence_frame))

        # 6. 置信度颜色：绿色>=75%，黄色>=50%，红色<50%
        conf_pct   = probs[idx1] * 100
        conf_color = (COLOR_SUCCESS if conf_pct >= 75
                      else (COLOR_WARN if conf_pct >= 50 else COLOR_DANGER))

        # 7. 显示预测正误对比（与数据库真实标签对比）
        correct_str = ""
        if ground_truth_label:
            is_correct  = (DISEASE_CLASSES[idx1] == ground_truth_label)
            correct_str = "  [Correct]" if is_correct else f"  [True: {ground_truth_label}]"

        summary = (
            f"Top-1: {DISEASE_CLASSES[idx1]}  ({conf_pct:.1f}%){correct_str}\n"
            f"Top-2: {DISEASE_CLASSES[idx2]}  ({probs[idx2]*100:.1f}%)"
        )

        # 8. 更新 UI，解锁 Step 3 按钮
        root_instance.after(0, lambda: (
            result_label.config(text=summary, foreground=conf_color),
            btn_generate_report.config(state=tk.NORMAL),
            status_label.config(
                text="Step 2 complete. Click GENERATE REPORT to proceed to Step 3.")
        ))

        # 9. 预加载音频供播放
        pygame.mixer.music.load(full_path)

    except Exception as e:
        root_instance.after(0, lambda: (
            messagebox.showerror("CNN Error", str(e)),
            btn_predict.config(state=tk.NORMAL),
            status_label.config(text=f"CNN Error: {e}")
        ))


def on_predict_clicked():
    """
    Step 2 按钮回调：在后台线程启动 CNN 推理（不阻塞 UI 主线程）。
    音频路径从路径输入框读取（用户通过 Browse 按钮选择）。
    ground_truth 通过 pid 从数据库反查，不依赖下拉框固定标签。
    """
    full_path = audio_path_var.get().strip()
    if not full_path or not os.path.isfile(full_path):
        messagebox.showerror("Error", "Please browse and select a valid .wav file first.")
        return

    btn_predict.config(state=tk.DISABLED)
    btn_generate_report.config(state=tk.DISABLED)
    result_label.config(text="Running CNN inference...", foreground="gray")
    status_label.config(text="Step 2: CNN inference in progress...")

    # 从文件名解析 pid，再从数据库反查真实标签（用于显示预测正误）
    pid          = extract_pid_from_filename(full_path)
    ground_truth = ground_truth_from_db(pid)

    def task():
        run_cnn_inference(full_path, ground_truth, root)
        root.after(0, lambda: btn_predict.config(state=tk.NORMAL))

    threading.Thread(target=task, daemon=True).start()


def on_browse_clicked():
    """
    Browse 按钮回调：打开文件对话框，默认定位到 audio_and_txt_files 目录。
    只允许选择 .wav 文件，选择后更新路径输入框并在状态栏显示 pid 和真实标签。
    """
    init_dir = PATH_AUDIO_DIR if os.path.isdir(PATH_AUDIO_DIR) else os.getcwd()
    path = filedialog.askopenfilename(
        title="Select a respiratory audio file (.wav)",
        initialdir=init_dir,
        filetypes=[("WAV files", "*.wav"), ("All files", "*.*")]
    )
    if path:
        audio_path_var.set(path)
        # 自动解析 pid 和真实标签，更新状态栏
        pid    = extract_pid_from_filename(path)
        gt     = ground_truth_from_db(pid)
        gt_str = f"DB label: {gt}" if gt else "PID not in DB"
        status_label.config(
            text=f"Selected: {os.path.basename(path)}  |  PID: {pid}  |  {gt_str}")


def on_play_clicked():
    """播放 / 停止音频（需先完成 Step 2）"""
    if pygame.mixer.music.get_busy():
        pygame.mixer.music.stop()
    else:
        try:
            pygame.mixer.music.play()
        except Exception:
            messagebox.showwarning("Warning", "Please run CNN prediction first to load audio.")

# ============================================================
# 9. LLM Prompt 构建
#    CNN 输出（疾病预测 + 置信度）是第一输入
#    置信度三档影响措辞风格：
#      >= 75% -> HIGH：语气较确定
#      50-75% -> MODERATE：建议进一步检查
#      < 50%  -> LOW：明确标注不确定性
# ============================================================
def build_llm_prompt(lang):
    """
    根据 cnn_result 构建 LLM prompt。
    涵盖：CNN 预测、置信度注释、听诊统计、全库基准对比、鉴别诊断提示。
    """
    disease     = cnn_result["predicted_disease"]
    conf_pct    = cnn_result["confidence"] * 100
    pid         = cnn_result["pid"]
    runner_up   = cnn_result["runner_up_disease"]
    runner_conf = cnn_result["runner_up_conf"] * 100

    # 置信度分层措辞
    if conf_pct >= 75:
        certainty_note = (f"Model confidence is HIGH ({conf_pct:.1f}%). "
                          "Use this finding as a primary clinical indicator pending confirmation.")
    elif conf_pct >= 50:
        certainty_note = (f"Model confidence is MODERATE ({conf_pct:.1f}%). "
                          "Treat as a preliminary finding; further examination recommended.")
    else:
        certainty_note = (f"Model confidence is LOW ({conf_pct:.1f}%). "
                          "This result is uncertain. Specialist referral strongly advised.")

    # 听诊统计（有 pid 则从数据库查，否则标 N/A）
    if pid is not None and not final_df.empty:
        c_total, w_total = get_patient_sounds(pid)
        sound_info = (f"Patient auscultation: {c_total} crackles, {w_total} wheezes "
                      f"(aggregated from all recorded cycles for PID {pid})")
    else:
        c_total, w_total = "N/A", "N/A"
        sound_info = "Auscultation counts: Not available (no matching PID in database)"

    # 全库基准对比
    _, benchmarks, _ = get_clinical_benchmarks()
    if disease in benchmarks.index:
        avg_c = benchmarks.loc[disease, 'crackles']
        avg_w = benchmarks.loc[disease, 'wheezes']
        benchmark_info = (f"Institutional avg for {disease}: "
                          f"{avg_c} crackles/cycle, {avg_w} wheezes/cycle")
    else:
        avg_c, avg_w   = "N/A", "N/A"
        benchmark_info = "Institutional benchmark: Not available"

    prompt = f"""ROLE: Senior Respiratory Consultant at UCL Hospital.

IMPORTANT DISCLAIMER: This report is generated from an AI CNN audio classification model.
This is NOT a confirmed clinical diagnosis. Always use language like "AI-predicted", "suggests", or "consistent with".

PATIENT DATA:
- Patient ID: {pid if pid else 'Unknown'}
- AI-Predicted Diagnosis: {disease}
- {certainty_note}
- Runner-up diagnosis: {runner_up} ({runner_conf:.1f}%) - consider as differential diagnosis
- {sound_info}
- {benchmark_info}

STRICT RULES:
1. DO NOT invent vital signs, lab values, or imaging results.
2. If confidence < 50%, prominently flag uncertainty in BOTH sections.
3. If patient has 0 sounds, write "Clear lung fields on auscultation".
4. Always contrast patient sounds against institutional average.

OUTPUT in {lang}. Use EXACTLY this format with the separator:

[PATIENT_SECTION]
- Condition Summary: Explain AI finding in patient-friendly language; compare sounds to benchmark.
- Home Care Orders: Rest, hydration, medication adherence advice.
- Precautions & Follow-up: When to seek emergency care; re-assessment schedule.
===SPLIT===
[WARD_SECTION]
Subject: AI-Assisted Internal Clinical Directive - Patient {pid if pid else 'Unknown'}
[CLINICAL OVERVIEW] Contrast {c_total} crackles / {w_total} wheezes vs {disease} benchmark ({avg_c}/{avg_w}). Note confidence.
[DIFFERENTIAL NOTE] Explain why {runner_up} ({runner_conf:.1f}%) should or should not be considered.
[WARD ORDERS] Proposed clinical interventions for AI-predicted {disease}.
[NURSING PLAN] Monitoring q6h, positioning, patient education.
[RED FLAGS] Specific escalation triggers for this condition.
"""
    return prompt

# ============================================================
# 10. LLM 报告生成
# ============================================================
def on_generate_report_clicked():
    """
    Step 3 按钮回调：读取 cnn_result，构建 prompt，异步调用智谱 AI，
    解析 ===SPLIT=== 分隔符，分别填入患者报告和病区指令文本框。
    """
    if cnn_result["predicted_disease"] is None:
        messagebox.showwarning("Warning", "Please run CNN prediction first (Step 2).")
        return

    lang = lang_var.get()
    btn_generate_report.config(state=tk.DISABLED)
    txt_patient.delete(1.0, tk.END)
    txt_ward.delete(1.0, tk.END)
    txt_patient.insert(tk.END, f"Generating patient report ({lang})...\n[Model: {LLM_MODEL}]")
    txt_ward.insert(tk.END, f"Generating ward directive ({lang})...\n[Model: {LLM_MODEL}]")
    status_label.config(text="Step 3: LLM report generation in progress...")

    prompt = build_llm_prompt(lang)

    def run():
        try:
            response  = client.chat.completions.create(
                model    = LLM_MODEL,
                messages = [{"role": "user", "content": prompt}]
            )
            full_text = response.choices[0].message.content.strip()

            # 用 ===SPLIT=== 分割患者报告和病区指令
            parts        = full_text.split("===SPLIT===", 1)
            patient_text = parts[0].replace("[PATIENT_SECTION]", "").strip().lstrip('#').strip()
            ward_text    = (parts[1].replace("[WARD_SECTION]", "").strip().lstrip('#').strip()
                            if len(parts) > 1 else full_text)

            root.after(0, lambda: (
                txt_patient.delete(1.0, tk.END),
                txt_patient.insert(tk.END, patient_text),
                txt_ward.delete(1.0, tk.END),
                txt_ward.insert(tk.END, ward_text),
                btn_generate_report.config(state=tk.NORMAL),
                status_label.config(text="Step 3 complete. Report generated.")
            ))
        except Exception as e:
            root.after(0, lambda: (
                messagebox.showerror("LLM Error", str(e)),
                btn_generate_report.config(state=tk.NORMAL),
                status_label.config(text=f"LLM Error: {e}")
            ))

    threading.Thread(target=run, daemon=True).start()

# ============================================================
# 11. 导出功能（PDF / TXT）
# ============================================================
def export_report(get_content_fn, mode="txt"):
    """
    将文本框内容导出为 PDF 或 TXT。
    PDF 模式优先使用 SimHei（支持中文），不存在则回退 Arial。
    """
    content = get_content_fn()
    ext  = ".pdf" if mode == "pdf" else ".txt"
    path = filedialog.asksaveasfilename(defaultextension=ext)
    if not path:
        return
    if mode == "pdf":
        try:
            from fpdf import FPDF
            pdf = FPDF()
            pdf.add_page()
            font_path = r"C:\Windows\Fonts\simhei.ttf"
            if os.path.exists(font_path):
                pdf.add_font('SimHei', '', font_path)
                pdf.set_font('SimHei', size=11)
            else:
                pdf.set_font("Arial", size=10)
            pdf.multi_cell(0, 8, txt=safe_text(content))
            pdf.output(path)
            messagebox.showinfo("Success", "PDF saved.")
        except Exception as e:
            messagebox.showerror("PDF Error", str(e))
    else:
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        messagebox.showinfo("Success", "Text file saved.")

# ============================================================
# 12. GUI 主布局：三步流程统一界面
# ============================================================
def launch_unified_gui():
    """
    构建并启动统一端到端 GUI。
    布局：左侧三步控制侧栏 + 右侧（上：三图并排可视化 + 下：双面板报告）
    """
    global root, audio_path_var, waveform_frame, spectrogram_frame, confidence_frame
    global result_label, btn_predict, btn_generate_report, status_label
    global txt_patient, txt_ward, lang_var

    root = tk.Tk()
    root.title("UCL Respiratory Intelligence v7.0 - CNN->LLM End-to-End")
    root.geometry("1620x970")
    root.configure(bg=COLOR_BG)

    header = tk.Frame(root, bg=COLOR_UCL_BLUE, height=65)
    header.pack(fill=tk.X)
    tk.Label(
        header,
        text="UCL RESPIRATORY INTELLIGENCE v7.0  --  CNN -> LLM  END-TO-END SYSTEM",
        bg=COLOR_UCL_BLUE, fg="white", font=("Segoe UI", 13, "bold")
    ).place(x=340, y=18)

    body = tk.Frame(root, bg=COLOR_BG)
    body.pack(fill=tk.BOTH, expand=True)

    # ========================
    # 左侧控制侧栏
    # ========================
    sidebar = tk.Frame(body, bg=COLOR_SIDEBAR, width=265)
    sidebar.pack(side=tk.LEFT, fill=tk.Y)
    sidebar.pack_propagate(False)

    def sidebar_section(text):
        tk.Label(sidebar, text=text, bg=COLOR_SIDEBAR, fg="white",
                 font=("Segoe UI", 9, "bold")).pack(pady=(18, 5), padx=15, anchor="w")

    def sidebar_btn(text, color, cmd, state=tk.NORMAL):
        b = tk.Button(sidebar, text=text, bg=color, fg="white",
                      font=("Segoe UI", 8, "bold"), relief="flat",
                      cursor="hand2", command=cmd, state=state)
        b.pack(fill=tk.X, padx=25, pady=4, ipady=6)
        return b

    # STEP 1: 浏览并选择音频文件
    sidebar_section("-- STEP 1: SELECT AUDIO --")

    # 路径输入框
    audio_path_var = tk.StringVar(value="")
    path_entry = tk.Entry(
        sidebar, textvariable=audio_path_var,
        font=("Segoe UI", 7), bg="#353B48", fg="white",
        relief="flat", state="readonly")
    path_entry.pack(fill=tk.X, padx=25, pady=(4, 2))

    # Browse 按钮
    tk.Button(
        sidebar, text="Browse audio_and_txt_files...",
        bg=COLOR_ACCENT, fg="white",
        font=("Segoe UI", 8, "bold"), relief="flat", cursor="hand2",
        command=on_browse_clicked
    ).pack(fill=tk.X, padx=25, pady=(0, 4))

    tk.Label(sidebar, text="(PID & ground truth auto-resolved from DB)",
             bg=COLOR_SIDEBAR, fg="#718093", font=("Segoe UI", 7)).pack()

    # STEP 2: CNN 推理
    sidebar_section("-- STEP 2: CNN INFERENCE --")
    btn_predict = sidebar_btn("RUN CNN PREDICTION", COLOR_ACCENT, on_predict_clicked)
    sidebar_btn("PLAY / STOP AUDIO", "#718093", on_play_clicked)

    # STEP 3: LLM 报告生成（初始禁用，需先完成 Step 2）
    sidebar_section("-- STEP 3: LLM REPORT --")
    tk.Label(sidebar, text="Report Language:", bg=COLOR_SIDEBAR,
             fg="#A2ABBD", font=("Segoe UI", 8)).pack(anchor="w", padx=25)
    lang_var = tk.StringVar(value="English")
    ttk.Combobox(sidebar, textvariable=lang_var,
                 values=["English", "Chinese", "Portuguese"],
                 state="readonly", width=23).pack(fill=tk.X, padx=25, pady=4)
    btn_generate_report = sidebar_btn(
        "GENERATE REPORT", COLOR_UCL_BLUE, on_generate_report_clicked, state=tk.DISABLED)

    # 分隔线
    tk.Frame(sidebar, bg="#4b5563", height=1).pack(fill=tk.X, padx=25, pady=14)

    # 导出区域
    sidebar_section("-- EXPORT --")
    sidebar_btn("SAVE PATIENT REPORT (PDF)", COLOR_SUCCESS,
                lambda: export_report(lambda: txt_patient.get(1.0, tk.END), "pdf"))
    sidebar_btn("SAVE WARD DIRECTIVE  (TXT)", "#8C7AE6",
                lambda: export_report(lambda: txt_ward.get(1.0, tk.END), "txt"))

    # ========================
    # 右侧内容区
    # ========================
    right = tk.Frame(body, bg=COLOR_BG)
    right.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=10, pady=8)

    # 上半：三图并排可视化
    viz_row = tk.Frame(right, bg=COLOR_BG)
    viz_row.pack(fill=tk.X)

    def make_viz_card(parent, title):
        card = tk.Frame(parent, bg=COLOR_CARD, bd=0,
                        highlightbackground="#DCDDE1", highlightthickness=1)
        card.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5, pady=4)
        tk.Label(card, text=title, bg=COLOR_CARD, fg=COLOR_UCL_BLUE,
                 font=("Segoe UI", 9, "bold")).pack(pady=5)
        inner = tk.Frame(card, bg=COLOR_CARD, height=170)
        inner.pack(fill=tk.BOTH, expand=True)
        return inner

    waveform_frame    = make_viz_card(viz_row, "WAVEFORM")
    spectrogram_frame = make_viz_card(viz_row, "MEL SPECTROGRAM")
    confidence_frame  = make_viz_card(viz_row, "CNN CONFIDENCE  (All 6 Classes)")

    # CNN 结果文字标签
    result_label = tk.Label(
        right,
        text="Browse a WAV file (Step 1)  ->  RUN CNN PREDICTION (Step 2)  ->  GENERATE REPORT (Step 3)",
        bg=COLOR_BG, fg="gray", font=("Segoe UI", 10, "italic"))
    result_label.pack(pady=3)

    # 下半：双面板文本报告
    report_row = tk.Frame(right, bg=COLOR_BG)
    report_row.pack(fill=tk.BOTH, expand=True, pady=(4, 0))

    def make_report_card(parent, title):
        card = tk.Frame(parent, bg=COLOR_CARD, bd=0,
                        highlightbackground="#DCDDE1", highlightthickness=1)
        card.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5)
        tk.Label(card, text=title, bg=COLOR_CARD, fg=COLOR_UCL_BLUE,
                 font=("Segoe UI", 9, "bold")).pack(pady=6)
        txt = scrolledtext.ScrolledText(
            card, font=("Segoe UI", 9), bg=COLOR_CARD, relief="flat", padx=8, pady=6)
        txt.pack(fill=tk.BOTH, expand=True)
        return txt

    txt_patient = make_report_card(
        report_row, "PATIENT CORRESPONDENCE  [AI-Assisted - Not for Direct Clinical Use]")
    txt_ward = make_report_card(
        report_row, "WARD DIRECTIVE  [AI-Assisted]")

    # 底部状态栏
    status_bar = tk.Frame(root, bg=COLOR_UCL_BLUE, height=26)
    status_bar.pack(fill=tk.X, side=tk.BOTTOM)
    status_label = tk.Label(
        status_bar,
        text="Ready - Browse a .wav file to begin",
        bg=COLOR_UCL_BLUE, fg="#A2ABBD", font=("Segoe UI", 8))
    status_label.pack(side=tk.LEFT, padx=10, pady=4)

    root.mainloop()


# 启动统一端到端 GUI
if __name__ == "__main__":
    launch_unified_gui()
